# Lesson 5: Underfitting and Overfitting

## Introduction

Now that you have a way to measure model accuracy, you can experiment with different models to find the best one.

But which models should you try? The answer comes from understanding **overfitting** and **underfitting**.

## Overfitting

A model **overfits** when it captures patterns in the training data that don't generalize to new data.

Example: A decision tree with very many leaves (splits) fits the training data almost perfectly, but performs poorly on validation data.

## Underfitting

A model **underfits** when it fails to capture important patterns in the data — even in training data.

Example: A decision tree with very few leaves is too simple to capture the patterns in the data.

## The Sweet Spot

```
Validation Error
     |
     |\                        
     |  \       (sweet spot)   
     |    \   /‾‾‾‾‾‾‾‾‾‾‾‾‾  
     |      \/                 
     |________________________
       Underfitting  Overfitting
                Model Complexity
```

We want a model that is complex enough to learn the real patterns, but not so complex that it memorizes the training data.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Recreate the dataset
data = {
    'Rooms': [2, 2, 3, 3, 4, 4, 2, 3, 4, 2, 3, 4, 2, 3, 4, 2, 3, 4, 2, 3,
              2, 3, 4, 2, 3, 4, 2, 3, 4, 2, 3, 4, 2, 3, 4, 2, 3, 4, 2, 3],
    'Distance': [2.5, 2.5, 2.5, 2.5, 2.5, 3.0, 3.0, 3.5, 4.0, 4.5,
                 2.8, 3.2, 3.7, 4.1, 2.6, 2.9, 3.3, 3.8, 4.2, 2.7,
                 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5,
                 10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5],
    'Bathroom': [1, 1, 2, 2, 1, 2, 1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 1, 2, 1, 2,
                 1, 1, 2, 1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 2],
    'Landsize': [202, 156, 134, 94, 120, 181, 245, 256, 300, 190,
                 210, 175, 230, 265, 310, 185, 240, 270, 195, 220,
                 180, 160, 200, 220, 250, 280, 320, 190, 210, 175,
                 300, 340, 260, 290, 310, 170, 230, 260, 285, 195],
    'BuildingArea': [79, 79, 150, 142, 210, 175, 95, 160, 220, 85,
                     130, 185, 100, 165, 225, 90, 140, 195, 80, 155,
                     110, 135, 175, 90, 155, 210, 85, 120, 170, 95,
                     200, 240, 145, 180, 215, 88, 132, 178, 200, 105],
    'YearBuilt': [1900, 1900, 1900, 2014, 2014, 2010, 1950, 1970, 2000, 1980,
                  1990, 2005, 1960, 1975, 2008, 1955, 1985, 2012, 1945, 1995,
                  2003, 1968, 1992, 2010, 1982, 1957, 2015, 1972, 1998, 2007,
                  1963, 2001, 1988, 1977, 2016, 1948, 1993, 2009, 1985, 1971],
    'Price': [1480000, 1035000, 1465000, 850000, 1600000, 1876000, 1636000, 1097000, 1350000, 750000,
              1200000, 1550000, 900000, 1100000, 1700000, 800000, 1300000, 1450000, 700000, 1050000,
              1150000, 980000, 1380000, 820000, 1250000, 1500000, 720000, 1010000, 1320000, 880000,
              1600000, 1800000, 1120000, 1280000, 1520000, 760000, 1180000, 1420000, 1080000, 930000]
}

melbourne_data = pd.DataFrame(data)
feature_columns = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Distance']
X = melbourne_data[feature_columns]
y = melbourne_data.Price

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

## Comparing Different Tree Sizes

The `max_leaf_nodes` parameter controls the complexity of the tree.

In [ ]:
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    """Return MAE for a decision tree with the given max_leaf_nodes."""
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=1)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return mae

# Compare MAE for different tree sizes
results = {}
for max_leaf_nodes in [2, 5, 10, 20, 50, 100]:
    mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    results[max_leaf_nodes] = mae
    print(f"Max leaf nodes: {max_leaf_nodes:>4}  |  MAE: ${mae:>12,.0f}")

best_size = min(results, key=results.get)
print(f"\nBest max_leaf_nodes: {best_size} (MAE: ${results[best_size]:,.0f})")

## Fitting the Final Model with Optimal Size

In [ ]:
# Fit the final model with the best tree size using ALL available data
final_model = DecisionTreeRegressor(max_leaf_nodes=best_size, random_state=1)
final_model.fit(X, y)
print(f"Final model trained with max_leaf_nodes={best_size}")

## Summary

| Problem | Description | Fix |
|---------|-------------|-----|
| **Overfitting** | Model is too complex; memorizes training data | Reduce tree depth / fewer leaves |
| **Underfitting** | Model is too simple; misses patterns | Increase tree depth / more leaves |

- Use the **validation MAE** to compare models with different complexities.
- The `max_leaf_nodes` parameter is a useful way to control tree complexity.

**Next up → Lesson 6: Random Forests**